In [4]:
# Mount Google Drive to access datasets
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
# Import required libraries
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')


In [13]:
# Load datasets from your Google Drive
# Replace the path with your actual file location
sentiment_df = pd.read_csv('/content/fear_greed_index.csv')
trader_df = pd.read_csv('/content/historical_data.csv')

# Display first few rows
print("Sentiment Dataset:")
print(sentiment_df.head())
print("\nTrader Dataset:")
print(trader_df.head())


Sentiment Dataset:
    timestamp  value classification        date
0  1517463000     30           Fear  2018-02-01
1  1517549400     15   Extreme Fear  2018-02-02
2  1517635800     40           Fear  2018-02-03
3  1517722200     24   Extreme Fear  2018-02-04
4  1517808600     11   Extreme Fear  2018-02-05

Trader Dataset:
                                      Account  Coin  Execution Price  \
0  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9769   
1  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9800   
2  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9855   
3  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9874   
4  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9894   

   Size Tokens  Size USD Side     Timestamp IST  Start Position Direction  \
0       986.87   7872.16  BUY  02-12-2024 22:50        0.000000       Buy   
1        16.00    127.68  BUY  02-12-2024 22:50      986.524596       Buy   
2       144.

In [14]:
# Check dataset information
print("Sentiment Dataset Info:")
print(sentiment_df.info())
print("\n" + "="*50 + "\n")
print("Trader Dataset Info:")
print(trader_df.info())

# Check for missing values
print("\nMissing values in Sentiment Dataset:")
print(sentiment_df.isnull().sum())
print("\nMissing values in Trader Dataset:")
print(trader_df.isnull().sum())

print('*'*70)
print(f"\nSentiment Dataset Shape: {sentiment_df.shape}")
print(f"Trader Dataset Shape: {trader_df.shape}")
print(f"\nDate Range - Sentiment: {sentiment_df['date'].min()} to {sentiment_df['date'].max()}")
print(f"Unique Traders: {trader_df['Account'].nunique():,}")
print(f"Unique Coins Traded: {trader_df['Coin'].nunique()}")


Sentiment Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2644 entries, 0 to 2643
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   timestamp       2644 non-null   int64 
 1   value           2644 non-null   int64 
 2   classification  2644 non-null   object
 3   date            2644 non-null   object
dtypes: int64(2), object(2)
memory usage: 82.8+ KB
None


Trader Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211224 entries, 0 to 211223
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Account           211224 non-null  object 
 1   Coin              211224 non-null  object 
 2   Execution Price   211224 non-null  float64
 3   Size Tokens       211224 non-null  float64
 4   Size USD          211224 non-null  float64
 5   Side              211224 non-null  object 
 6   Timestamp IST     211224 no

In [17]:
# Rename columns for consistency
sentiment_df.columns = ['timestamp', 'value', 'classification', 'date']
trader_df.columns = [col.lower().replace(' ', '_') for col in trader_df.columns]

# Convert date columns
sentiment_df['date'] = pd.to_datetime(sentiment_df['date'])
# For the format "18-03-2025 12:50"
trader_df['timestamp_ist'] = pd.to_datetime(trader_df['timestamp_ist'],
                                             format='%d-%m-%Y %H:%M')
trader_df['date'] = trader_df['timestamp_ist'].dt.date
trader_df['date'] = pd.to_datetime(trader_df['date'])


# Handle missing values intelligently
print("\n" + "="*70)
print("HANDLING MISSING VALUES")
print("="*70)

# Fee and Trade ID missing values are acceptable - fill with 0
trader_df['fee'] = trader_df['fee'].fillna(0)
trader_df['trade_id'] = trader_df['trade_id'].fillna('unknown')

# Drop rows with critical missing values
trader_df = trader_df.dropna(subset=['closed_pnl', 'execution_price', 'size_usd'])

print(f"Cleaned Trader Dataset Shape: {trader_df.shape}")


HANDLING MISSING VALUES
Cleaned Trader Dataset Shape: (211224, 17)


In [18]:
# Convert date columns to datetime format
sentiment_df['Date'] = pd.to_datetime(sentiment_df['date'])
trader_df['timestamp_ist'] = pd.to_datetime(trader_df['timestamp_ist'])

# Create a 'Date' column in trader dataset (extract date from timestamp)
trader_df['Date'] = trader_df['timestamp_ist'].dt.date
trader_df['Date'] = pd.to_datetime(trader_df['Date'])

print("Date conversion completed!")
print(sentiment_df['Date'].dtype)
print(trader_df['Date'].dtype)

Date conversion completed!
datetime64[ns]
datetime64[ns]


In [19]:
# ADVANCED FEATURE ENGINEERING
print("\n" + "="*70)
print("ADVANCED FEATURE ENGINEERING")
print("="*70)

# 1. Time-based features (critical for temporal analysis)
trader_df['hour'] = trader_df['timestamp_ist'].dt.hour
trader_df['day_of_week'] = trader_df['timestamp_ist'].dt.dayofweek
trader_df['is_weekend'] = trader_df['day_of_week'].isin([5, 6]).astype(int)
trader_df['month'] = trader_df['timestamp_ist'].dt.month
trader_df['quarter'] = trader_df['timestamp_ist'].dt.quarter

# 2. Trading session classification
def classify_trading_session(hour):
    if 6 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 18:
        return 'Afternoon'
    elif 18 <= hour < 24:
        return 'Evening'
    else:
        return 'Night'

trader_df['trading_session'] = trader_df['hour'].apply(classify_trading_session)

# 3. Profit/Loss metrics
trader_df['pnl_category'] = pd.cut(trader_df['closed_pnl'],
                                     bins=[-np.inf, -100, 0, 100, np.inf],
                                     labels=['Large Loss', 'Small Loss', 'Small Profit', 'Large Profit'])

trader_df['is_profitable'] = (trader_df['closed_pnl'] > 0).astype(int)
trader_df['pnl_percentage'] = (trader_df['closed_pnl'] / trader_df['size_usd']) * 100

# 4. Risk metrics
trader_df['position_size_category'] = pd.cut(trader_df['size_usd'],
                                              bins=[0, 1000, 5000, 20000, np.inf],
                                              labels=['Small', 'Medium', 'Large', 'Very Large'])

# 5. Trade characteristics
trader_df['is_long'] = (trader_df['side'] == 'BUY').astype(int)
trader_df['abs_pnl'] = trader_df['closed_pnl'].abs()

print("✓ Time-based features created")
print("✓ Trading session classification completed")
print("✓ P&L metrics calculated")
print("✓ Risk categorization completed")



ADVANCED FEATURE ENGINEERING
✓ Time-based features created
✓ Trading session classification completed
✓ P&L metrics calculated
✓ Risk categorization completed


In [20]:
# Merge datasets
merged_df = pd.merge(trader_df, sentiment_df[['date', 'classification', 'value']],
                     on='date', how='left')

# Fill any missing sentiment values with forward fill
merged_df['classification'] = merged_df['classification'].fillna(method='ffill')
merged_df['value'] = merged_df['value'].fillna(method='ffill')

# Create sentiment intensity categories
def categorize_sentiment_intensity(row):
    if row['classification'] == 'Extreme Fear':
        return 'Extreme Fear'
    elif row['classification'] == 'Fear':
        return 'Fear'
    elif row['classification'] == 'Greed':
        return 'Greed'
    elif row['classification'] == 'Extreme Greed':
        return 'Extreme Greed'
    else:
        return 'Neutral'

merged_df['sentiment_intensity'] = merged_df.apply(categorize_sentiment_intensity, axis=1)

# Binary sentiment classification
merged_df['is_fear'] = merged_df['classification'].str.contains('Fear', na=False).astype(int)
merged_df['is_greed'] = merged_df['classification'].str.contains('Greed', na=False).astype(int)

print(f"\n✓ Merged Dataset Shape: {merged_df.shape}")
print(f"✓ Sentiment Coverage: {(merged_df['classification'].notna().sum() / len(merged_df) * 100):.1f}%")



✓ Merged Dataset Shape: (211224, 35)
✓ Sentiment Coverage: 100.0%


In [21]:
# CALCULATE ADVANCED TRADER-LEVEL METRICS
print("\n" + "="*70)
print("CALCULATING TRADER-LEVEL METRICS")
print("="*70)

# Aggregate trader performance metrics
trader_metrics = merged_df.groupby('account').agg({
    'closed_pnl': ['sum', 'mean', 'std', 'count'],
    'size_usd': ['sum', 'mean'],
    'is_profitable': 'mean',  # Win rate
    'abs_pnl': 'mean',
    'is_long': 'mean'  # Long bias
}).reset_index()

trader_metrics.columns = ['account', 'total_pnl', 'avg_pnl', 'pnl_volatility',
                         'trade_count', 'total_volume', 'avg_position_size',
                         'win_rate', 'avg_abs_pnl', 'long_bias']

# Calculate Sharpe-like ratio (risk-adjusted returns)
trader_metrics['risk_adjusted_return'] = trader_metrics['avg_pnl'] / (trader_metrics['pnl_volatility'] + 1)

# Classify traders by performance
trader_metrics['trader_type'] = pd.cut(trader_metrics['total_pnl'],
                                        bins=[-np.inf, -500, 0, 500, 2000, np.inf],
                                        labels=['Heavy Loser', 'Loser', 'Breakeven',
                                               'Winner', 'Heavy Winner'])

# Merge trader metrics back to main dataset
merged_df = pd.merge(merged_df, trader_metrics[['account', 'win_rate', 'trader_type',
                                                 'risk_adjusted_return']],
                     on='account', how='left')

print(f"✓ Trader metrics calculated for {trader_metrics.shape[0]:,} unique traders")



CALCULATING TRADER-LEVEL METRICS
✓ Trader metrics calculated for 32 unique traders


In [22]:
# ROLLING WINDOW ANALYSIS (time-series feature engineering)
print("\n" + "="*70)
print("TIME SERIES FEATURE ENGINEERING")
print("="*70)

# Sort by date for time-series operations
merged_df = merged_df.sort_values(['account', 'timestamp_ist'])

# Calculate rolling statistics per trader (7-day window)
merged_df['rolling_pnl_7d'] = merged_df.groupby('account')['closed_pnl'].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean()
)

merged_df['rolling_volume_7d'] = merged_df.groupby('account')['size_usd'].transform(
    lambda x: x.rolling(window=7, min_periods=1).sum()
)

# Calculate cumulative metrics
merged_df['cumulative_pnl'] = merged_df.groupby('account')['closed_pnl'].cumsum()
merged_df['trade_number'] = merged_df.groupby('account').cumcount() + 1

# Momentum indicator (recent performance vs historical)
merged_df['pnl_momentum'] = merged_df.groupby('account')['closed_pnl'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean()
)

print("✓ Rolling window features created")
print("✓ Cumulative metrics calculated")
print("✓ Momentum indicators added")



TIME SERIES FEATURE ENGINEERING
✓ Rolling window features created
✓ Cumulative metrics calculated
✓ Momentum indicators added


In [24]:
# Save processed datasets
import os

# Create the directory if it doesn't exist
output_dir = '/content/drive/MyDrive/csv_files'
os.makedirs(output_dir, exist_ok=True)

merged_df.to_csv(f'{output_dir}/merged_data_advanced.csv', index=False)
trader_metrics.to_csv(f'{output_dir}/trader_metrics.csv', index=False)

print("\n" + "="*70)
print("DATA PREPARATION COMPLETE")
print("="*70)
print(f"✓ Final dataset shape: {merged_df.shape}")
print(f"✓ Features created: {len(merged_df.columns)} columns")
print(f"✓ Files saved to Google Drive")


DATA PREPARATION COMPLETE
✓ Final dataset shape: (211224, 43)
✓ Features created: 43 columns
✓ Files saved to Google Drive
